# TUNE1 DQN_256 — 무리 세계에서 처음부터 DQN, 은닉층 256-256

선생님·계획기 없이 빈 신경망에서 학습합니다. 난수 3개 × 600판을 학습한 뒤, 모델마다 84상황을 잽니다. 방법과 읽는 법은 `results/explore/tune1_20261105/사전등록.md`에 고정했습니다.

칸 6개(`TUNE1_DQN_*`, `TUNE1_PPO_*`)는 **노트북 하나에 런타임 하나**입니다. 동시에 돌려도 되고, 런타임이 비는 대로 차례로 돌려도 됩니다.

1. 런타임 유형 **CPU + 높은 RAM(CPU 8개)** 을 고르고 **런타임 → 모두 실행**을 누르세요. GPU는 필요 없습니다.
2. 이 노트북 하나가 끝까지 돕니다: 점검(약 10분) → 학습(약 7~9시간, 어림) → 평가(약 1시간, 어림).
3. 끊기면 그 노트북만 다시 **모두 실행**하세요. 학습은 마지막 체크포인트부터, 평가는 남은 판부터 이어 갑니다.
4. 칸 6개와 `TUNE1_baseline`이 모두 끝나면 `TUNE1_judge`를 실행하고 "끝났어"라고 알려 주세요. 결과: Drive `FutureTraffic/explore_tune/tune1/`.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, time

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_TUNE1.zip"
WORKSPACE = Path("/content/FutureTraffic_TUNE1")
PREP = DRIVE / "explore_nn3" / "prep"                  # N5(읽기만, 해시 대조)
OUT = DRIVE / "explore_tune" / "tune1"
CELL = "DQN_256"
MIN_CPUS = 8
for path in (ARCHIVE, PREP / "start" / "seed_42" / "model.pt", OUT / "baseline" / "runs"):
    assert path.exists(), f"필요한 파일·폴더가 없습니다: {path}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 칸:", CELL, "| 결과 폴더:", OUT)
assert CPUS >= MIN_CPUS, (f"CPU가 {CPUS}개입니다. 런타임 → 런타임 유형 변경에서 CPU와 '높은 RAM'을 고른 뒤 다시 실행하세요.")

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "TUNE1_V1", "TUNE1 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("tune1_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| PLATOON0 설정 {package_manifest['platoon0_config_sha256'][:12]} | 측정 계획 {package_manifest['plan_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
# ARENA3 심판 기준선과 같은 평가 지문이 되도록 같은 패키지 버전을 씁니다.
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
os.chdir(WORKSPACE)
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", "test_explore_tune.py"], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 실행 전 점검(약 10분): 작은 학습 → 저장 모델 평가 → 신호 대조. 통과해야 본 학습으로 갑니다.
CHECK = OUT / CELL / "check"
result = CHECK / "check.json"
if not (colab.read_json(result) or {}).get("ok"):
    colab.run_live(["src/explore_tune.py", "check", "--cell", CELL, "--prep", str(PREP), "--out", str(CHECK), "--work", "/content/tc"], WORKSPACE, check=False)
r = colab.read_json(result)
assert r and r.get("ok"), f"점검 실패 — 본 실행을 하지 않습니다. 이 칸의 출력을 저에게 보내 주세요: {r}"
print("점검 통과 | 맥과 같음:", r.get("same_as_mac"), "| 평가 한 판 초:", round(r["eval_seconds"]), "| 작은 학습 작업시간:", round(r["train"]["cpu_hours"], 3))

In [ ]:
# 본 학습: 난수 3개 × 판 2개 동시. 백그라운드로 돌아 셀을 멈춰도 계속 돕니다. 끊기면 처음부터 다시 모두 실행하세요.
colab.run_live(["src/explore_tune.py", "prepare", "--cell", CELL, "--out", str(OUT)], WORKSPACE)
LOGS = OUT / CELL / "logs"
TRAIN = OUT / CELL / "train"

def describe(name):
    s = colab.read_json(TRAIN / f"seed_{name.rsplit('_', 1)[1]}" / "status.json") or {}
    return f"{s.get('episodes', 0)}/{s.get('budget', 600)}판, 결정 {s.get('decisions', 0)}, 최근 지연 {s.get('recent_delay_s')}, 버림 {s.get('discarded', 0)}, 작업시간 {s.get('cpu_hours', 0):.1f}h"

jobs = {f"train_{seed}": ["src/explore_tune.py", "train", "--cell", CELL, "--seed", str(seed), "--out", str(OUT), "--work", f"/content/tt{seed}"] for seed in (42, 43, 44)}
pids = colab.launch(jobs, WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 300, describe)
assert code == 0, "오류가 났습니다. 위 로그를 저에게 보내 주세요(다시 실행하면 체크포인트부터 이어 갑니다)."
print("학습 끝.")

In [ ]:
# 평가: 최종 모델 3개 × 84상황 + 곡선 모델 9개 × 12상황(360판). 끝난 판은 건너뜁니다.
def describe_eval(name, folder=OUT / CELL / "eval" / "runs"):
    return f"끝난 판 {len(list(folder.glob('*/*/metrics.json')))}/360"

pids = colab.launch({"eval": ["src/explore_tune.py", "evaluate", "--cell", CELL, "--out", str(OUT), "--work", "/content/te", "--workers", str(CPUS)]}, WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 300, describe_eval)
assert code == 0, "평가에서 오류가 났습니다. 위 로그를 저에게 보내 주세요(다시 실행하면 끝난 판은 건너뜁니다)."
print("이 칸 끝. 칸 6개와 TUNE1_baseline이 끝나면 TUNE1_judge를 실행하세요.")